<a href="https://colab.research.google.com/github/samhoon000/Job-Portal-Database-Analysis/blob/main/Linkedin_Job_Posting_Cleaning_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
df=pd.read_csv("postings.csv", engine='python',on_bad_lines='skip')

In [21]:
df.head()

,job_id,company_name,title,description,max_salary,pay_period,location,company_id,views,med_salary,...,skills_desc,listed_time,posting_domain,sponsored,work_type,currency,compensation_type,normalized_salary,zip_code,fips
0,921716,Corcoran Sawyer Smith,Marketing Coordinator,Job descriptionA leading real estate firm in N...,20.0,HOURLY,"Princeton, NJ",2774458.0,20.0,NaN,...,Requirements: \n\nWe are seeking a College or ...,1.713398e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,38480.0,8540.0,34021.0
1,1829192,NaN,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...",50.0,HOURLY,"Fort Collins, CO",NaN,1.0,NaN,...,NaN,1.712858e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,83200.0,80521.0,8069.0
2,10998357,The National Exemplar,Assitant Restaurant Manager,The National Exemplar is accepting application...,65000.0,YEARLY,"Cincinnati, OH",64896719.0,8.0,NaN,...,We are currently accepting resumes for FOH - A...,1.713278e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,55000.0,45202.0,39061.0
3,23221523,"Abrams Fensterman, LLP",Senior Elder Law / Trusts and Estates Associat...,Senior Associate Attorney - Elder Law / Trusts...,175000.0,YEARLY,"New Hyde Park, NY",766262.0,16.0,NaN,...,This position requires a baseline understandin...,1.712896e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,157500.0,11040.0,36059.0
4,35982263,NaN,Service Technician,Looking for HVAC service tech with experience ...,80000.0,YEARLY,"Burlington, IA",NaN,3.0,NaN,...,NaN,1.713452e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,70000.0,52601.0,19057.0


In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123849 entries, 0 to 123848
Data columns (total 31 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   job_id                      123849 non-null  int64  
 1   company_name                122130 non-null  object 
 2   title                       123849 non-null  object 
 3   description                 123842 non-null  object 
 4   max_salary                  29793 non-null   float64
 5   pay_period                  36073 non-null   object 
 6   location                    123849 non-null  object 
 7   company_id                  122132 non-null  float64
 8   views                       122160 non-null  float64
 9   med_salary                  6280 non-null    float64
 10  min_salary                  29793 non-null   float64
 11  formatted_work_type         123849 non-null  object 
 12  applies                     23320 non-null   float64
 13  original_liste

In [23]:
df=df.drop(columns=['job_posting_url','application_url','zip_code','fips','expiry','closed_time'])

In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123849 entries, 0 to 123848
Data columns (total 25 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   job_id                      123849 non-null  int64  
 1   company_name                122130 non-null  object 
 2   title                       123849 non-null  object 
 3   description                 123842 non-null  object 
 4   max_salary                  29793 non-null   float64
 5   pay_period                  36073 non-null   object 
 6   location                    123849 non-null  object 
 7   company_id                  122132 non-null  float64
 8   views                       122160 non-null  float64
 9   med_salary                  6280 non-null    float64
 10  min_salary                  29793 non-null   float64
 11  formatted_work_type         123849 non-null  object 
 12  applies                     23320 non-null   float64
 13  original_liste

In [25]:
cols_to_drop = [
    'med_salary',        # ~95% missing
    'skills_desc',       # almost empty
    'remote_allowed',    # too sparse + not critical
    'currency',          # tied to salary (but salary already weak)
    'compensation_type', # sparse
    'pay_period',        # sparse
    'posting_domain',    #not useful for analysis
    'normalized_salary'  #~70% missing → unreliable
]

In [26]:
df = df.drop(columns=cols_to_drop)

In [27]:
df['views'] = df['views'].fillna(0)
df['applies'] = df['applies'].fillna(0)
df['min_salary'] = df['min_salary'].fillna(0)
df['max_salary'] = df['max_salary'].fillna(0)

In [34]:
df=df.dropna(subset=['company_name','description'])
df['company_id']=df['company_id'].fillna(-1)
df['formatted_experience_level'] = df['formatted_experience_level'].fillna('Not Specified')

In [36]:
df['listed_time'] = pd.to_datetime(df['listed_time'], unit='ms')

In [40]:
df['original_listed_time'] = pd.to_datetime(df['original_listed_time'], unit='ms')

In [41]:
df.head()

,job_id,company_name,title,description,max_salary,location,company_id,views,min_salary,formatted_work_type,applies,original_listed_time,application_type,formatted_experience_level,listed_time,sponsored,work_type
0,921716,Corcoran Sawyer Smith,Marketing Coordinator,Job descriptionA leading real estate firm in N...,20.0,"Princeton, NJ",2774458.0,20.0,17.0,Full-time,2.0,2024-04-17 23:45:08,ComplexOnsiteApply,Not Specified,2024-04-17 23:45:08,0,FULL_TIME
2,10998357,The National Exemplar,Assitant Restaurant Manager,The National Exemplar is accepting application...,65000.0,"Cincinnati, OH",64896719.0,8.0,45000.0,Full-time,0.0,2024-04-16 14:26:54,ComplexOnsiteApply,Not Specified,2024-04-16 14:26:54,0,FULL_TIME
3,23221523,"Abrams Fensterman, LLP",Senior Elder Law / Trusts and Estates Associat...,Senior Associate Attorney - Elder Law / Trusts...,175000.0,"New Hyde Park, NY",766262.0,16.0,140000.0,Full-time,0.0,2024-04-12 04:23:32,ComplexOnsiteApply,Not Specified,2024-04-12 04:23:32,0,FULL_TIME
5,91700727,Downtown Raleigh Alliance,Economic Development and Planning Intern,Job summary:The Economic Development & Plannin...,20.0,"Raleigh, NC",1481176.0,9.0,14.0,Internship,4.0,2024-04-18 16:01:39,ComplexOnsiteApply,Not Specified,2024-04-18 16:01:39,0,INTERNSHIP
6,103254301,Raw Cereal,Producer,Company DescriptionRaw Cereal is a creative de...,300000.0,United States,81942316.0,7.0,60000.0,Contract,1.0,2024-04-11 18:43:39,SimpleOnsiteApply,Not Specified,2024-04-11 18:43:39,0,CONTRACT
